# 15 — The Empirical Residual-Correction Map

Even with a fully-fitted 15-coefficient harmonic distortion model, real detectors leave a residual systematic deviation between predicted and observed ring positions — typically 50–100 µε of per-pixel drift that the analytical basis can't capture. The fix, ported from v1 C's `dg_residual_corr_lookup`, is an **empirical smooth ΔR(Y, Z) map** stored at detector resolution and applied via bilinear lookup after parallax.

This notebook covers:

1. What the residual map *is* (a 2-D ΔR field in pixels).
2. **Building** it from per-fit residuals at MAP (thin-plate-spline RBF).
3. **Applying** it through the differentiable `pixel_to_REta` forward.
4. The before / after impact on calibrant strain — typically pushes 100–200 µε down to <30 µε.
5. Persisting / loading via the v1-compatible binary format that all three MIDAS packages share.

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, h5py, numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

## 1. Differentiable bilinear lookup

The lookup is `torch.nn.functional.grid_sample` under the hood — autograd-clean in both the query coordinates and the map values. So the map can be **fixed** (a smooth empirical correction built once post-MAP) **or refined as a Parameter** in joint inverse problems.

In [ ]:
from midas_calibrate_v2.forward.residual_corr import residual_corr_lookup

# Toy map: ΔR(Y, Z) = 0.3 * sin(2π Y / 1000) (pixels)
Nz, Ny = 512, 512
z, y = np.meshgrid(np.arange(Nz), np.arange(Ny), indexing='ij')
demo_map = 0.3 * np.sin(2*np.pi * y / 1000.0)
demo_map = torch.as_tensor(demo_map, dtype=torch.float64)

# Query at a few pixel positions; grad flows through both args.
Y = torch.tensor([100.5, 250.3, 400.1], dtype=torch.float64, requires_grad=True)
Z = torch.tensor([200.0, 300.0, 100.5], dtype=torch.float64, requires_grad=True)
vals = residual_corr_lookup(Y, Z, demo_map)
vals.sum().backward()
print('lookup values  :', vals.detach().numpy())
print('dY gradients   :', Y.grad.numpy())
print('dZ gradients   :', Z.grad.numpy())

## 2. Build from per-fit residuals (the post-MAP step)

After the LM has converged on geometry + analytical distortion, we collect each fit's $\Delta R = R_\text{forward} - R_\text{ideal}$ (µm), drop outliers, and fit a thin-plate-spline RBF over the (Y, Z) field. The RBF is evaluated on a coarse grid and upsampled to full detector resolution. Net cost ~ a few seconds even on 2880².

In [ ]:
from midas_calibrate_v2.forward.residual_corr import build_residual_corr_map

# Synthetic: pretend 400 fitted peaks have ΔR = 10*sin(2π Y/2048) µm.
rng = np.random.default_rng(seed=0)
N = 400
Yp = rng.uniform(100.0, 1900.0, size=N)
Zp = rng.uniform(100.0, 1900.0, size=N)
dR_um = 10.0 * np.sin(2*np.pi * Yp / 2048.0) + rng.normal(0, 1.0, size=N)

cm = build_residual_corr_map(
    torch.as_tensor(Yp), torch.as_tensor(Zp), torch.as_tensor(dR_um),
    NrPixelsY=2048, NrPixelsZ=2048, pxY=200.0,
)
print(f'map shape={tuple(cm.shape)}  dtype={cm.dtype}')
print(f'range [{cm.min():.3f}, {cm.max():.3f}] px, rms={float((cm**2).mean()**0.5):.3f} px')

## 3. End-to-end: real CeO₂ image, strain before / after

The `calibrate()` entry point builds and applies the map automatically. To see the impact, we run it twice — once with `build_residual_corr=False` and once with `True` — on the same image.

In [ ]:
DATA = Path(os.environ.get('V2_ONESHOT_FILE',
    '/Users/hsharma/Desktop/analysis/leighanne/data/CeO2_XRD_echem_JLi_002587.vrx.h5'))
if not DATA.exists():
    raise FileNotFoundError(f'set V2_ONESHOT_FILE; missing {DATA}')
with h5py.File(DATA, 'r') as f:
    img  = f['exchange/data'][0].astype(np.float32)
    dark = f['exchange/data_dark'][0].astype(np.float32)

from midas_calibrate_v2 import calibrate
common = dict(image=img, wavelength=0.184139, pxY=150.0, dark=dark,
              calibrant='CeO2', verbose=False)

r_no  = calibrate(**common, build_residual_corr=False, output_dir='./calib_no_map')
r_yes = calibrate(**common, build_residual_corr=True,  output_dir='./calib_with_map')

print(f'WITHOUT residual map:  in-loop strain = {r_no.in_loop_strain_uE:.1f} µε  '
      f'post = {r_no.post_residual_strain_uE}')
print(f'WITH    residual map:  in-loop strain = {r_yes.in_loop_strain_uE:.1f} µε  '
      f'post = {r_yes.post_residual_strain_uE:.1f} µε  ← typically ~3-5× lower')

## 4. What does the map look like?

Smooth field, units of pixels. Usually shows azimuthal structure (the un-modelled higher-fold harmonics) and a slowly-varying radial trend (residual parallax / panel skew). RMS in the 0.05–0.3 px range is normal.

In [ ]:
cm = r_yes.residual_corr_map.cpu().numpy()
vlim = float(np.percentile(np.abs(cm), 99))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
ax.plot(r_yes.BC_y, r_yes.BC_z, 'ko', ms=6)
ax.set_title(f'ΔR(Y, Z) at MAP, pixels\n'
             f'rms={float((cm**2).mean()**0.5):.3f} px, '
             f'range [{cm.min():.3f}, {cm.max():.3f}] px')
plt.colorbar(im, ax=ax, label='ΔR (px)'); plt.tight_layout(); plt.show()

## 5. Round-trip via v1-compatible binary

The `residual_corr.bin` written by `calibrate()` is the same wire format as v1 C's `dg_residual_corr_lookup` expects: `NrPixelsY × NrPixelsZ` little-endian float64 in `map[z * Ny + y]` order. Loadable by:

* `midas_calibrate_v2.forward.residual_corr.load_residual_corr_bin` (torch)
* `midas_integrate.residual_corr.load_residual_correction_map` (numpy, legacy)
* `midas_integrate_v2` via `IntegrationSpec.ResidualCorrectionMap = '<path>'`
* C `CalibrantIntegratorOMP` via `ResidualCorrMapFN`

In [ ]:
from midas_calibrate_v2.forward.residual_corr import (
    save_residual_corr_bin, load_residual_corr_bin,
)
save_residual_corr_bin(r_yes.residual_corr_map, '/tmp/demo_map.bin')
rt = load_residual_corr_bin('/tmp/demo_map.bin',
                              NrPixelsY=r_yes.NrPixelsY,
                              NrPixelsZ=r_yes.NrPixelsZ)
print('roundtrip max diff:', float((rt - r_yes.residual_corr_map).abs().max()))

## When to use it

* **Always** for production calibrations — adds ~5 s, removes the 50–100 µε floor.
* **Don't** rebuild between sample exposures within the same beamtime — the map is a detector property, not a sample property. Save once, reuse.
* **Refit** if you replace a detector module, repointing the beamstop, or rebooting the detector electronics (rare 1–2× per beamtime).